# Dependencies (for venv)

In [ ]:
pip install -r requirements.txt

# Generate the dataset

In [ ]:
import random
import torch

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   
# torch.set_default_device(device)

# print(device)

In [ ]:
def generate_objects(N=3, block_size=0.4, initial_tower=True):
    # 1. Generate random order for the goals
    idx_goals = [i for i in range(0, N)]
    random.shuffle(idx_goals)

    # 2. Generate initial order for the blocks by inverting the order of goals
    idx_objs = idx_goals[::-1]

    # 3. Generate goals
    # https://www.generationrobots.com/media/panda-franka-emika-datasheet.pdf
    # according to the datasheet, the Franka Panda can reach in an area of 0.855 [m] around itself
    max_reach_dist = 0.7 # [m]

    # select a random (x,y) position for the goal.
    x_goal = random.random() * max_reach_dist
    y_goal = random.random() * max_reach_dist

    pos_goals = [(x_goal, y_goal, i*block_size) for i in range(0, N)]

    # Generate initial positions
    if initial_tower:
        x_obj = random.random() * max_reach_dist
        y_obj = random.random() * max_reach_dist

        pos_objects = [(x_obj, y_obj, i*block_size) for i in range(0, N)]
    else:
        pos_objects = [(random.random() * max_reach_dist, random.random() * max_reach_dist, i*block_size) for i in range(0, N)]

    goals = {}
    objects = {}
    for i in range(0, N):
        goals[idx_goals[i]] = pos_goals[i]
        objects[idx_objs[i]] = pos_objects[i]

    print(f"goals (in desired order of final stacking):\n{goals}")
    print(f"initial object positions (in order of stacking):\n{objects}")

    return objects, goals, idx_objs, idx_goals


In [ ]:
# Now solve the stacking problem by taking the highest block in the objects tower (last in order)
# and putting it into the lowest available goal (first in order)

# build the state vector, ordered nodes with categorical, position and binary features, followed by ordered goals with the same features
def build_state(objects, goals):
    s = []
    N = len(objects)
    for n in range(0, N):
        s.append(0) # objects are all blocks (one hot encode)
        # s.append(1) # objects are all blocks (one hot encode)


        s.append(int(objects[n] == goals[n])) # whether block is in goal

        
        for o in objects[n]: s.append(o)
        # s.append(objects[n]) # position of block

    for n in range(0, N):
        s.append(1) # objects are all blocks (one hot encode)
        # s.append(0) # objects are all blocks (one hot encode)


        s.append(int(objects[n] == goals[n])) # whether block is in goal


        # s.append(goals[n]) # position of block
        for o in goals[n]: s.append(o)

    return s

def build_example(N=3, block_size=0.2, initial_tower=True):
    objects, goals, idx_objs, idx_goals = generate_objects(N, block_size, initial_tower)
    
    tau = []    
    tau.append(build_state(objects, goals))

    for n in range(0, N):
        # print(f"step #{n}")
        
        o = N-1-n

        # print(f"choosing object #{idx_objs[o]} with pos {objects[idx_objs[o]]}")
        # print(f"moving object #{idx_objs[o]} to goal{goals[idx_objs[o]]}")

        objects[idx_objs[o]] = goals[idx_objs[o]]
        tau.append([o, o])

        # print(f"Updated object positions with goals:")
        # print(f"goals:\n{goals}")
        # print(f"current objects pos:\n{objects}")
        # print("\n")

        tau.append(build_state(objects, goals))

    # TODO (?): add the empty action to a complete graph
    # print(tau)
    return tau

In [ ]:
# Alternatevely
# at each step, find the highest block that is not yet in its goal
# and move it to the lowest available goal
# repeat until each blocks is in its goal

# Observation to graph

In [ ]:
# Now put the demonstration in graph format, using pytorch geometric

import torch
import torch.nn.functional as F
import torch_geometric
from torch_geometric.utils.convert import to_networkx
import networkx as nx

def build_dataset(N=3, N_EXAMPLES=50):
    data_list = []
    for i in range(N_EXAMPLES):
        # print(f"Building example #{i}")
        tau = build_example(N)

        states = tau[0:-1:2] # exclude the complete graph
        actions = tau[1::2]

        # print(len(states), states)
        # print(len(states), actions)

        # build edges, only once
        # graphs are always fully connected, undirected, and with no edge features and no self loops
        edges = torch.tensor([(i,j) for i in range(N*2) for j in range(N*2) if i != j]).t().contiguous()

        # build nodes and labels
        for n in range(0, N):
            s = states[n]
            a = actions[n]

            # torch_geometric.data.Data is a single graph
            # x is the input graph, as a matrix of dimension [node, node_features]
            # I have N*2 nodes (N blocks, N goals) and 5 features each
            x = torch.reshape(torch.tensor(s), (N*2, -1)) # N*2 considers both goals and blocks
            
            # additive measurement noise
            # z = torch.zeros(x.shape)
            # z[:, 2:] = torch.rand((z[:, 2:].shape))*0.005
            # x += z

            # y is divided into two vectors, one hot encoded, probabilities for the block and the goal
            y = torch.Tensor.float(F.one_hot(torch.tensor(a), num_classes=N)).t()
            y = (y[:, 0].reshape(-1), y[:, 1].reshape(-1))
            print(y)
            # y = torch.flatten(y)
            # print(y)

            # pos = x[:, 2:]
            # x = x[:, :2]
            
            # y = torch.tensor(a)
            G = torch_geometric.data.Data(x=x, edge_index=edges, y=y)

            # save all the graphs into a single data_list for use as a DataLoader later
            data_list.append(G)

        # print(data_list)
    return data_list

# Build torch_geometric dataset from examples

In [ ]:
# https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_dataset.html#frequently-asked-questions
# just create a DataLoader starting from the list of example Data
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

train_dataset = build_dataset(N=3, N_EXAMPLES=50) + build_dataset(N=4, N_EXAMPLES=50)
# train_dataset = build_dataset(N=4, N_EXAMPLES=30) + build_dataset(N=3, N_EXAMPLES=30)
# print("Training dataset: ", train_dataset)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(build_dataset(N=3, N_EXAMPLES=10) + build_dataset(N=4, N_EXAMPLES=10), batch_size=1, shuffle=True)
test_loader = DataLoader(build_dataset(N=3, N_EXAMPLES=10), batch_size=1)


# Build NN

In [ ]:
# https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html
# https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html
# https://colab.research.google.com/github/Alireza-Akhavan/graph-neural-network/blob/main/07-Graph_Classification_pyg.ipynb#scrollTo=CN3sRVuaQ88l
# https://khairulislam.github.io/GNN/pages/graph/Graph_Classification.html

# regarding the use of position (and generalization to unseen positions): https://github.com/pyg-team/pytorch_geometric/issues/3326
# regarding fully connected graphs and the use of dense layers: https://github.com/pyg-team/pytorch_geometric/issues/964

from torch.nn import Linear
from torch_geometric.nn import DenseSAGEConv, DenseGATConv
from torch_geometric.utils import to_dense_adj, to_dense_batch, to_nested_tensor

import lightning as L
from lightning.pytorch.callbacks import Callback, EarlyStopping
from lightning.pytorch.callbacks.early_stopping import EarlyStoppingReason

class GCN(torch.nn.Module):
    def __init__(self, hidden_nodes=64,dropout_p=0.5):
        super().__init__()
        # All policies consist of 3 hidden layers, with 64 hidden units each and ReLU activation.
        # For attention policies, the number of attention heads were set to 1.
        self.conv1 = DenseSAGEConv(5, hidden_nodes, normalize=True)
        self.conv2 = DenseSAGEConv(hidden_nodes, hidden_nodes, normalize=True)
        self.conv3 = DenseSAGEConv(hidden_nodes, hidden_nodes, normalize=True)
        # self.conv1 = DenseGATConv(5, hidden_nodes, dropout=dropout_p)
        # self.conv2 = DenseGATConv(hidden_nodes, hidden_nodes, dropout=dropout_p)
        # self.conv3 = DenseGATConv(hidden_nodes, hidden_nodes, dropout=dropout_p)
        # self.lin1 = Linear(hidden_nodes, hidden_nodes, bias=True)
        self.lin2 = Linear(hidden_nodes, 1, bias=True)
        self.dropout_p = dropout_p

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch # batch is the index of the example

        # some noise. Probably wrong to be added here but simpler than in batch
        
        # print(data.num_nodes)
        # print("x", x, x.shape)
        # print("edge_index", edge_index)
        # print("batch", batch)

        # x, b = to_dense_batch(x, batch, max_num_nodes=1)
        # print("Dense batch")
        # print(x, b)
        # print(x.shape, b.shape)

        # additive measurement noise during training
        # if self.training:
            # z = torch.zeros(x.shape)
            # z[:, 2:] = torch.rand((z[:, 2:].shape))*0.005
            # x += z

        adj = to_dense_adj(edge_index)
        # print(x, x.shape)
        # print(adj, adj.shape)
        x = self.conv1(x, adj)
        x = F.relu(x)
        # x = F.dropout(x, p=self.dropout_p, training=self.training)
        x = self.conv2(x, adj)
        x = F.relu(x)
        # x = F.dropout(x, p=self.dropout_p, training=self.training)
        x = self.conv3(x, adj)
        x = F.relu(x)
        # x = F.dropout(x, p=self.dropout_p, training=self.training)
        # x = self.lin1(x)
        # x = F.relu(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)
        x = self.lin2(x)

        # The output of the GNN policy is K + L dimensional, reshaped into K and L dimensional outputs
        objects = x[:, :data.num_nodes//2]
        goals = x[:, data.num_nodes//2:]

        return objects, goals
        # return F.softmax(objects, dim=1), F.softmax(goals, dim=1)

model = GCN(hidden_nodes=64, dropout_p=0.0) #.to(device)
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(3000):
    running_loss = 0.

    model.train()
    loss = torch.zeros((1))
    optimizer.zero_grad()
    for i, batch in enumerate(train_loader):
        objects, goals = model(batch)
        # print(batch)
        objects = objects.squeeze()
        goals = goals.squeeze()
        # print(objects, objects.shape, goals, goals.shape)
        # print(batch.y)
        # print(batch.y[0], batch.y[0].shape)
        l1 = F.cross_entropy(objects, batch.y[0])
        l2 = F.cross_entropy(goals, batch.y[1])
        loss += l1+l2
        # loss += l1
        # out = model(batch)
        # print(out, out.shape)
        # print(batch.y, batch.y.shape)

        running_loss += loss.item()
        # loss = F.cross_entropy(out, batch.y.reshape(out.shape))

    loss.backward()
    optimizer.step()

    print(f'Training loss (CE): {loss.item()}')

    model.eval()
    correct_obj = 0.
    correct_goal = 0.
    for i,data in enumerate(train_loader):
        objects, goals = model(data)
        objects = objects.squeeze()
        goals = goals.squeeze()
        # print(torch.argmax(objects)==data.y[0], torch.argmax(objects), data.y[0])
        correct_obj += (torch.argmax(F.softmax(objects)) == torch.argmax(data.y[0])).float()
        correct_goal += (torch.argmax(F.softmax(goals)) == torch.argmax(data.y[1])).float()
    acc_obj = correct_obj / i
    acc_goal = correct_goal / i
    print(f'Training Accuracy (objects/goals): {acc_obj:.4f}/{acc_goal:.4f}')
    
    # if acc_obj > 0.5: print(objects, F.softmax(objects, dim=0))
    correct_obj = 0.
    correct_goal = 0.
    for i,data in enumerate(val_loader):
        objects, goals = model(data)
        objects = objects.squeeze()
        goals = goals.squeeze()
        # print(torch.argmax(objects)==data.y[0], torch.argmax(objects), data.y[0])
        correct_obj += (torch.argmax(F.softmax(objects)) == torch.argmax(data.y[0])).float()
        correct_goal += (torch.argmax(F.softmax(goals)) == torch.argmax(data.y[1])).float()
    acc_obj = correct_obj / i
    acc_goal = correct_goal / i
    print(f'Validation Accuracy (objects/goals): {acc_obj:.4f}/{acc_goal:.4f}')

    if acc_obj > 0.8 and acc_goal > 0.8: break

In [ ]:
model.eval()

test_loader = DataLoader(build_dataset(N=3, N_EXAMPLES=20), batch_size=1)

correct_obj = 0.
correct_goal = 0.
for i,data in enumerate(val_loader):
    objects, goals = model(data)
    objects = objects.squeeze()
    goals = goals.squeeze()
    # print(torch.argmax(objects)==data.y[0], torch.argmax(objects), data.y[0])
    correct_obj += (torch.argmax(F.softmax(objects)) == torch.argmax(data.y[0])).float()
    correct_goal += (torch.argmax(F.softmax(goals)) == torch.argmax(data.y[1])).float()
acc_obj = correct_obj / i
acc_goal = correct_goal / i
print(f'Test Accuracy (objects/goals): {acc_obj:.4f}/{acc_goal:.4f}')